In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..') / 'src'))
from config import CH

# Quality Check — Validaciones de Calidad
Ejecuta validaciones automáticas sobre las 3 capas.
- ✓ = validación pasada correctamente
- ✗ ALERTA = problema detectado que requiere revisión

In [2]:
def check_bronze(ch):
    print('=' * 50)
    print('VALIDACIONES BRONZE')
    print('=' * 50)

    checks = {
        'raw_rental': {
            'total':      'SELECT count() FROM bronze.raw_rental',
            'nulos_pk':   'SELECT count() FROM bronze.raw_rental WHERE rental_id = 0',
            'duplicados': 'SELECT count() FROM (SELECT rental_id, count() AS c FROM bronze.raw_rental GROUP BY rental_id HAVING c > 1)',
        },
        'raw_film': {
            'total':      'SELECT count() FROM bronze.raw_film',
            'nulos_pk':   'SELECT count() FROM bronze.raw_film WHERE film_id = 0',
            'duplicados': 'SELECT count() FROM (SELECT film_id, count() AS c FROM bronze.raw_film GROUP BY film_id HAVING c > 1)',
        },
        'raw_customer': {
            'total':      'SELECT count() FROM bronze.raw_customer',
            'nulos_pk':   'SELECT count() FROM bronze.raw_customer WHERE customer_id = 0',
            'duplicados': 'SELECT count() FROM (SELECT customer_id, count() AS c FROM bronze.raw_customer GROUP BY customer_id HAVING c > 1)',
        },
        'raw_payment': {
            'total':      'SELECT count() FROM bronze.raw_payment',
            'nulos_pk':   'SELECT count() FROM bronze.raw_payment WHERE payment_id = 0',
            'monto_cero': 'SELECT count() FROM bronze.raw_payment WHERE amount = 0',
        },
        'raw_address': {
            'total':      'SELECT count() FROM bronze.raw_address',
            'nulos_pk':   'SELECT count() FROM bronze.raw_address WHERE address_id = 0',
        },
        'raw_inventory': {
            'total':      'SELECT count() FROM bronze.raw_inventory',
            'nulos_pk':   'SELECT count() FROM bronze.raw_inventory WHERE inventory_id = 0',
        },
    }

    for tabla, validaciones in checks.items():
        print(f'\n── {tabla} ──')
        for nombre, query in validaciones.items():
            resultado = ch.execute(query)[0][0]
            estado    = 'OK' if resultado == 0 or nombre == 'total' else 'ALERTA'
            simbolo   = '✓' if estado == 'OK' else '✗'
            print(f'  {simbolo} {nombre:20} {resultado:>8,}   {estado if estado == "ALERTA" else ""}')

check_bronze(CH)

VALIDACIONES BRONZE

── raw_rental ──
  ✓ total                  16,044   
  ✓ nulos_pk                    0   
  ✓ duplicados                  0   

── raw_film ──
  ✓ total                   1,000   
  ✓ nulos_pk                    0   
  ✓ duplicados                  0   

── raw_customer ──
  ✓ total                     599   
  ✓ nulos_pk                    0   
  ✓ duplicados                  0   

── raw_payment ──
  ✓ total                  16,044   
  ✓ nulos_pk                    0   
  ✗ monto_cero                 24   ALERTA

── raw_address ──
  ✓ total                     603   
  ✓ nulos_pk                    0   

── raw_inventory ──
  ✓ total                   4,581   
  ✓ nulos_pk                    0   


In [3]:
def check_silver(ch):
    print('=' * 50)
    print('VALIDACIONES SILVER')
    print('=' * 50)

    checks = {
        'stg_rental': {
            'total':      'SELECT count() FROM silver.stg_rental',
            'nulos_pk':   'SELECT count() FROM silver.stg_rental WHERE rental_id = 0',
        },
        'stg_film': {
            'total':       'SELECT count() FROM silver.stg_film',
            'sin_titulo':  "SELECT count() FROM silver.stg_film WHERE title = ''",
            'precio_cero': 'SELECT count() FROM silver.stg_film WHERE rental_rate = 0',
        },
        'stg_customer': {
            'total':      'SELECT count() FROM silver.stg_customer',
            'sin_nombre': "SELECT count() FROM silver.stg_customer WHERE full_name = ' '",
        },
        'stg_payment': {
            'total':          'SELECT count() FROM silver.stg_payment',
            'monto_cero':     'SELECT count() FROM silver.stg_payment WHERE amount = 0',
            'monto_negativo': 'SELECT count() FROM silver.stg_payment WHERE amount < 0',
        },
        'stg_address': {
            'total':      'SELECT count() FROM silver.stg_address',
            'nulos_pk':   'SELECT count() FROM silver.stg_address WHERE address_id = 0',
        },
    }

    for tabla, validaciones in checks.items():
        print(f'\n── {tabla} ──')
        for nombre, query in validaciones.items():
            resultado = ch.execute(query)[0][0]
            estado    = 'OK' if resultado == 0 or nombre == 'total' else 'ALERTA'
            simbolo   = '✓' if estado == 'OK' else '✗'
            print(f'  {simbolo} {nombre:20} {resultado:>8,}   {estado if estado == "ALERTA" else ""}')

check_silver(CH)

VALIDACIONES SILVER

── stg_rental ──
  ✓ total                  16,044   
  ✓ nulos_pk                    0   

── stg_film ──
  ✓ total                   1,000   
  ✓ sin_titulo                  0   
  ✓ precio_cero                 0   

── stg_customer ──
  ✓ total                     599   
  ✓ sin_nombre                  0   

── stg_payment ──
  ✓ total                  16,044   
  ✗ monto_cero                 24   ALERTA
  ✓ monto_negativo              0   

── stg_address ──
  ✓ total                     603   
  ✓ nulos_pk                    0   


In [4]:
def check_gold(ch):
    print('=' * 50)
    print('VALIDACIONES GOLD')
    print('=' * 50)

    checks = {
        'dim_film': {
            'total':      'SELECT count() FROM gold.dim_film',
            'sin_titulo': "SELECT count() FROM gold.dim_film WHERE title = ''",
            'duplicados': 'SELECT count() FROM (SELECT film_key, count() AS c FROM gold.dim_film GROUP BY film_key HAVING c > 1)',
        },
        'dim_customer': {
            'total':      'SELECT count() FROM gold.dim_customer',
            'sin_nombre': "SELECT count() FROM gold.dim_customer WHERE full_name = ' '",
            'sin_pais':   "SELECT count() FROM gold.dim_customer WHERE country = ''",
        },
        'dim_date': {
            'total':      'SELECT count() FROM gold.dim_date',
            'duplicados': 'SELECT count() FROM (SELECT date_key, count() AS c FROM gold.dim_date GROUP BY date_key HAVING c > 1)',
        },
        'fact_rental': {
            'total':           'SELECT count() FROM gold.fact_rental',
            'huerfanos_film':  'SELECT count() FROM gold.fact_rental fr LEFT JOIN gold.dim_film f ON fr.film_key = f.film_key WHERE f.film_key = 0',
            'huerfanos_cust':  'SELECT count() FROM gold.fact_rental fr LEFT JOIN gold.dim_customer c ON fr.customer_key = c.customer_key WHERE c.customer_key = 0',
            'huerfanos_date':  'SELECT count() FROM gold.fact_rental fr LEFT JOIN gold.dim_date d ON fr.date_key = d.date_key WHERE d.date_key = 0',
            'monto_negativo':  'SELECT count() FROM gold.fact_rental WHERE amount < 0',
        },
    }

    for tabla, validaciones in checks.items():
        print(f'\n── {tabla} ──')
        for nombre, query in validaciones.items():
            resultado = ch.execute(query)[0][0]
            estado    = 'OK' if resultado == 0 or nombre == 'total' else 'ALERTA'
            simbolo   = '✓' if estado == 'OK' else '✗'
            print(f'  {simbolo} {nombre:25} {resultado:>8,}   {estado if estado == "ALERTA" else ""}')

check_gold(CH)

VALIDACIONES GOLD

── dim_film ──
  ✓ total                        1,000   
  ✓ sin_titulo                       0   
  ✓ duplicados                       0   

── dim_customer ──
  ✓ total                          599   
  ✓ sin_nombre                       0   
  ✓ sin_pais                         0   

── dim_date ──
  ✓ total                        1,095   
  ✓ duplicados                       0   

── fact_rental ──
  ✓ total                       16,044   
  ✓ huerfanos_film                   0   
  ✓ huerfanos_cust                   0   
  ✓ huerfanos_date                   0   
  ✓ monto_negativo                   0   
